# TradeSense · Component 03 · notebook 43: uncertainty and confidence labels for the Integrity Score (CPU, no training)

**Why.** An alert should say *how sure* the system is, not only *that* something is unusual. Two kinds of uncertainty can be measured from what is already saved:

1. **Rarity (conformal-style p-level).** For a bar's fused score `s`, `p = share of normal (genuine) bars with a score at least as large`. Shown as "rarer than 1 in 100 / 1,000 / 10,000 normal minutes". Two versions are checked: **static** (calibrated once on the genuine DEV out-of-fold bars) and **rolling** (calibrated on the trailing 7 days of the genuine series, exactly how the deployed engine thresholds work).
2. **Agreement.** How many of the three base models (AE, XGBoost, Transformer; each at its own T1c Alert rule, 2 episodes/day) are alarming on the same bar.

**Confidence label (declared before the run).** High = fusion Alert and >= 2 models agree and rolling p <= 0.001; Medium = fusion Alert and (>= 2 models agree or rolling p <= 0.001); Low = any other fusion Alert.

**Pre-registered hypotheses (CAL-3, fusion v0).**
- **H-U1 (static calibration travels):** the share of genuine CAL-3 bars with static p <= alpha lies within 0.5x-2x of alpha for alpha = 0.01 on both symbols. Reported at 0.05, 0.01, 0.001. If it fails, the engine must use the rolling calibration (it does).
- **H-U2 (agreement is informative):** among fusion Alert bars of the injected CAL-3 series, the share of bars inside a real injected event is higher when >= 2 models agree than when only one does, by >= 0.10, with the 95% block-bootstrap interval (240-bar blocks) above 0.
- **H-U3 (rarity is informative):** the share of Alert bars inside an event rises across the rolling p tiers (p <= 0.01, 0.001, 0.0001) without a drop larger than 0.01, and the top tier exceeds the lowest by >= 0.05.
- **H-U4 (labels are ordered):** the in-event share is ordered High >= Medium >= Low (no inversion larger than 0.01).
- **Stated limit:** CAL-3 injects far more events than real life (about 18% of bars are inside an event), so absolute in-event shares are inflated; only the ordering and differences are used.

TEST-3, TEST-B and every TGNN test day are untouched. Nothing here changes a frozen rule.

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json', 'declared_fusion_v0.json', 'declared_watch_v1.json']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 35 (patched) and 40 first'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py') + ('  [MOCK RUN]' if MOCK else ''))
ALPHAS = [0.05, 0.01, 0.001, 0.0001]; TIERS = [0.01, 0.001, 0.0001]

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944


## 1 · Data and calibration sets

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; DEV_LO, DEV_HI = 90720, 129600
DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = list(DF['models']); FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}
FILES = {'AE-ctx fmax': 'ae_ctx', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
if (OUT/'declared_ae_v2.json').exists(): FILES['AE-ctx fmax'] = 'ae_ctx_agg'
L1V = B.valid_mask(129600, DEV_LO, DEV_HI); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
VD = np.zeros(129600, bool); VD[DEV_LO + 1440:DEV_HI] = True; VD[:B.WARM] = False
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
SC = {}; FS = {}; EV = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files); FS[s] = {k: z[k] for k in ['GEN_AO', 'CAL_r0', 'OOF_GEN_L1']}
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); EV[s] = e[e.replica == 0].reset_index(drop=True)
    for m in MODELS:
        z = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); assert not any('TEST' in k for k in z.files); d = {}
        for k in ['GEN_L1', 'GEN_AO', 'CAL_r0']:
            x = z[k]; x = x[:, 0] if x.ndim == 2 else x; d[k] = nanw(x[:AOLEN] if k.startswith('CAL') else x)
        SC[(m, s)] = d
INEV = {}
for s in SYMBOLS:
    m_ = np.zeros(AOLEN, bool)
    for r in EV[s].itertuples(): m_[r.t0:r.t0 + r.dur] = True
    INEV[s] = m_
DW = json.load(open(OUT/'declared_watch_v1.json')); QA = DW['alert']['q']
ALERT = {}; AGREE = {}; GENALERT = {}
for s in SYMBOLS:
    thr = thr_roll(FS[s]['GEN_AO'], QA[s]); ALERT[s] = FS[s]['CAL_r0'] > thr; GENALERT[s] = FS[s]['GEN_AO'] > thr
    ag = np.zeros(AOLEN, int)
    for m in MODELS:
        q = B.t1_q(SC[(m, s)]['GEN_L1'], L1V, B.PRIMARY); ag += (SC[(m, s)]['CAL_r0'] > thr_roll(SC[(m, s)]['GEN_AO'], q)).astype(int)
    AGREE[s] = ag
# static calibration: genuine DEV out-of-fold fused scores
STAT = {}
for s in SYMBOLS:
    d = FS[s]['OOF_GEN_L1'][VD]; d = np.sort(d[np.isfinite(d)]); STAT[s] = d
def static_p(s, x): d = STAT[s]; return (1 + (len(d) - np.searchsorted(d, x, side='left'))) / (len(d) + 1)
# rolling calibration: trailing 7-day quantiles of the GENUINE fused series (T1c style), one threshold series per level
ROLL = {s: {a: thr_roll(FS[s]['GEN_AO'], 1 - a) for a in ALPHAS} for s in SYMBOLS}
def rolling_level(s, x, idx):
    lev = np.full(len(x), 1.0)
    for a in sorted(ALPHAS, reverse=True): lev = np.where(x > ROLL[s][a][idx], a, lev)
    return lev
say('fusion and base-model scores loaded; Alert bars (fusion, 2/day): ' + str({s: int(ALERT[s].sum()) for s in SYMBOLS}) + '; share of CAL-3 bars inside an injected event: ' + str({s: round(float(INEV[s].mean()), 3) for s in SYMBOLS}))

fusion and base-model scores loaded; Alert bars (fusion, 2/day): {'BTCUSDT': 2918, 'ETHUSDT': 2957}; share of CAL-3 bars inside an injected event: {'BTCUSDT': 0.124, 'ETHUSDT': 0.127}


## 2 · Unit tests

In [ ]:
d = np.sort(np.arange(10.0)); STAT['T'] = d
assert abs(static_p('T', 9.5) - 1 / 11) < 1e-12 and abs(static_p('T', -1) - 1.0) < 1e-12 and abs(static_p('T', 4.0) - (1 + 6) / 11) < 1e-12; del STAT['T']; say('PASS T01 static p-value = (1 + number of normal bars at least as large) / (n + 1)')
for s in SYMBOLS:
    for a, b in [(0.05, 0.01), (0.01, 0.001), (0.001, 0.0001)]:
        ok = np.isfinite(ROLL[s][a]) & np.isfinite(ROLL[s][b]); assert (ROLL[s][b][ok] >= ROLL[s][a][ok] - 1e-9).all()
ix_ = np.array([6000, 6001]); x = np.array([-1e9, 1e9]); assert (rolling_level('BTCUSDT', x, ix_) == np.array([1.0, 0.0001])).all(); say('PASS T02 rolling level: a huge score reaches the rarest tier, a tiny one stays at 1')
lv = rolling_level('BTCUSDT', FS['BTCUSDT']['GEN_AO'][6000:20000], np.arange(6000, 20000)); assert set(np.unique(lv)) <= set(ALPHAS + [1.0]); say('PASS T03 thresholds are nested (rarer level = higher threshold) and levels take only the declared values')
for s in SYMBOLS: assert AGREE[s].min() >= 0 and AGREE[s].max() <= 3
say('PASS T04 agreement counts lie between 0 and 3')
say('PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)')

PASS T01 static p-value = (1 + number of normal bars at least as large) / (n + 1)
PASS T02 rolling level: a huge score reaches the rarest tier, a tiny one stays at 1
PASS T03 thresholds are nested (rarer level = higher threshold) and levels take only the declared values
PASS T04 agreement counts lie between 0 and 3
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)


## 3 · Calibration, agreement, tiers and labels

In [ ]:
rows = []
for s in SYMBOLS:
    g = FS[s]['GEN_AO']; ok = CALV & np.isfinite(g)
    for a in ALPHAS[:3]:
        p = np.array([static_p(s, x) for x in g[ok][::1]]) if False else (1 + (len(STAT[s]) - np.searchsorted(STAT[s], g[ok], side='left'))) / (len(STAT[s]) + 1)
        rows.append({'sym': s, 'alpha': a, 'static: genuine share with p<=alpha': float((p <= a).mean()), 'ratio': float((p <= a).mean() / a), 'rolling: genuine share': float((rolling_level(s, g[ok], np.flatnonzero(ok)) <= a).mean()), 'rolling ratio': float((rolling_level(s, g[ok], np.flatnonzero(ok)) <= a).mean() / a)})
CAL = pd.DataFrame(rows); say(''); say('--- calibration on the genuine CAL-3 bars (1.0 would be perfect) ---'); say(CAL.round(3).to_string(index=False))
HU1 = bool(all(0.5 <= r <= 2 for r in CAL[CAL.alpha == 0.01].ratio))
# bar tables over the Alert bars of the injected series
tab = []
for s in SYMBOLS:
    ix = np.flatnonzero(ALERT[s] & CALV); x = FS[s]['CAL_r0'][ix]; lev = rolling_level(s, x, ix)
    tab.append(pd.DataFrame({'sym': s, 't': ix, 'in_event': INEV[s][ix], 'agree': AGREE[s][ix], 'level': lev}))
T = pd.concat(tab, ignore_index=True); T['agree2'] = T.agree >= 2
def label(r):
    if r.agree >= 2 and r.level <= 0.001: return 'High'
    if r.agree >= 2 or r.level <= 0.001: return 'Medium'
    return 'Low'
T['label'] = T.apply(label, axis=1)
def block_ci(mask_a, mask_b, y, t, n=NBOOT, blk=240, seed=0):
    ids = (t // blk).astype(int); u = np.unique(ids); rg = np.random.default_rng(seed); idx = {k: np.flatnonzero(ids == k) for k in u}; out = []
    for _ in range(n):
        pick = rg.choice(u, len(u)); ii = np.concatenate([idx[k] for k in pick]); a, b, yy = mask_a[ii], mask_b[ii], y[ii]
        if a.sum() and b.sum(): out.append(yy[a].mean() - yy[b].mean())
    return np.percentile(out, [2.5, 97.5])
y = T.in_event.to_numpy(); tt = T.t.to_numpy(); a2 = T.agree2.to_numpy()
sh2, sh1 = float(y[a2].mean()), float(y[~a2].mean()); ci = block_ci(a2, ~a2, y, tt); HU2 = bool(sh2 - sh1 >= 0.10 and ci[0] > 0)
tier_rows = []
for tr in TIERS: m_ = (T.level <= tr).to_numpy(); tier_rows.append({'rolling tier': f'p <= {tr}', 'Alert bars': int(m_.sum()), 'in-event share': float(y[m_].mean()) if m_.sum() else np.nan})
TR = pd.DataFrame(tier_rows); sh = TR['in-event share'].to_numpy(); HU3 = bool(np.all(np.diff(sh) >= -0.01) and sh[-1] - sh[0] >= 0.05)
lab = T.groupby('label').agg(bars=('in_event', 'size'), in_event_share=('in_event', 'mean')).reindex(['High', 'Medium', 'Low']); l = lab.in_event_share.to_numpy(); HU4 = bool(np.all(np.diff(l[np.isfinite(l)]) <= 0.01))
say(''); say(f'--- fusion Alert bars in the injected CAL-3 series: {len(T)} bars; share inside an event by agreement ---')
say(pd.DataFrame({'agreeing models': ['>= 2', '1 (or 0)'], 'bars': [int(a2.sum()), int((~a2).sum())], 'in-event share': [sh2, sh1]}).round(3).to_string(index=False)); say(f'difference {sh2 - sh1:+.3f}, 95% block-bootstrap CI {ci[0]:+.3f}..{ci[1]:+.3f}')
say(''); say('in-event share by rolling p tier'); say(TR.round(3).to_string(index=False)); say(''); say('in-event share by confidence label (High / Medium / Low)'); say(lab.round(3).to_string())
say(''); say(f'H-U1 static calibration within 0.5x-2x at alpha 0.01 on both symbols: {HU1}'); say(f'H-U2 agreement >= 2 models raises the in-event share by >= 0.10 with CI above 0: {HU2}'); say(f'H-U3 in-event share rises across rolling p tiers: {HU3}'); say(f'H-U4 labels ordered High >= Medium >= Low: {HU4}')


--- calibration on the genuine CAL-3 bars (1.0 would be perfect) ---
    sym  alpha  static: genuine share with p<=alpha  ratio  rolling: genuine share  rolling ratio
BTCUSDT  0.050                                0.040  0.792                   0.049          0.973
BTCUSDT  0.010                                0.006  0.593                   0.010          0.991
BTCUSDT  0.001                                0.000  0.370                   0.001          1.273
ETHUSDT  0.050                                0.049  0.987                   0.054          1.072
ETHUSDT  0.010                                0.012  1.201                   0.011          1.106
ETHUSDT  0.001                                0.002  1.574                   0.001          1.435

--- fusion Alert bars in the injected CAL-3 series: 5875 bars; share inside an event by agreement ---
agreeing models  bars  in-event share
           >= 2  3377           0.893
       1 (or 0)  2498           0.727
difference +0.166, 95% bloc

## 4 · Save and declare

In [ ]:
CAL.to_csv(OUT/'uncertainty_calibration_cal3.csv', index=False); TR.to_csv(OUT/'uncertainty_tiers_cal3.csv', index=False); lab.to_csv(OUT/'uncertainty_labels_cal3.csv')
DECL = {'status': 'DECLARED before TEST-3 / TEST-B: rarity (rolling p-level) and agreement based confidence labels for fusion Alert bars.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'rolling_levels': ALPHAS, 'rolling_calibration': 'trailing 7-day quantile of the genuine fused series (T1c in evaluation; the live engine uses its own trailing 7-day series, T1)', 'static_calibration_percentiles_dev_oof': {s: {str(a): float(np.quantile(STAT[s], 1 - a)) for a in ALPHAS} for s in SYMBOLS},
        'agreement': 'number of models (AE, XGBoost, Transformer) above their own T1c Alert rule (2 episodes/day)', 'labels': {'High': 'fusion Alert, >= 2 models agree, rolling p <= 0.001', 'Medium': 'fusion Alert and (>= 2 models agree or rolling p <= 0.001)', 'Low': 'any other fusion Alert'},
        'hypotheses': {'H_U1': HU1, 'H_U2': HU2, 'H_U3': HU3, 'H_U4': HU4}, 'fusion_declared_sha16': sha_file(OUT/'declared_fusion_v0.json')}
(OUT/'declared_uncertainty_v1.json').write_text(json.dumps(DECL, indent=1))
block = '\n'.join(LINES); (OUT/'b2_step14_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step14_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'b2_step14_share_block.txt')


===== SHARE BLOCK (attach models_b2_v1/b2_step14_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
fusion and base-model scores loaded; Alert bars (fusion, 2/day): {'BTCUSDT': 2918, 'ETHUSDT': 2957}; share of CAL-3 bars inside an injected event: {'BTCUSDT': 0.124, 'ETHUSDT': 0.127}
PASS T01 static p-value = (1 + number of normal bars at least as large) / (n + 1)
PASS T02 rolling level: a huge score reaches the rarest tier, a tiny one stays at 1
PASS T03 thresholds are nested (rarer level = higher threshold) and levels take only the declared values
PASS T04 agreement counts lie between 0 and 3
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)

--- calibration on the genuine CAL-3 bars (1.0 would be perfect) ---
    sym  alpha  static: genuine share with p<=alpha  ratio  rolling: genuine share  rolling ratio
BTCUSDT  0.050                                0.040  0.792        